# DP-SGD at ten seeds

The corrected sweep in Section 5.11 runs three seeds, where the
smallest two-sided p a paired signed-rank test can return is 0.25 --
so neither the ECE difference favouring AHA-Fed nor the accuracy
difference favouring FedAvg is resolvable by construction. Every
other comparison in the paper uses ten seeds.

This raises it to ten at both budgets. run_one skips completed
files, so a resumed session reuses whatever is already on disk.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist 'opacus==1.5.4'

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    # opacus binds to the torch it was built against, so reinstall the
    # pinned version after any swap.
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    '--force-reinstall','--no-deps','opacus==1.5.4'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')

# Fail fast if the DP stack is broken, and say which versions are in play:
# version 1 died here on opacus importing DTensor from a torch that has none.
CHECK=('import torch, opacus;'
       'print("opacus", opacus.__version__, "torch", torch.__version__);'
       'from opacus import PrivacyEngine;'
       'print("PrivacyEngine OK")')
r=subprocess.run([sys.executable,'-c',CHECK],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-800:])
if r.returncode!=0: raise SystemExit('opacus import failed')


In [ ]:
import traceback
from paper.run_real_experiments import run_one, OUT_DIR

# ten seeds, matching every other comparison in the paper
SEEDS=[42,123,456,7,13,21,99,2024,31337,555]
EPS=[1.0,10.0]   # opposite ends of the budget range answer the question

# The paper's standard budget: 1000 img/client, 25 rounds, against the
# 300/15 the original DP sweep used. Batch 128 is deliberate -- DP-SGD
# trades batch size against noise at fixed epsilon.
done=fail=0
for eps in EPS:
    dp=dict(epsilon=eps, delta=1e-5, max_grad_norm=1.0)
    for s in SEEDS:
        common=dict(dataset='pathmnist', num_classes=9,
                    dp_config=dp, num_rounds=25, samples_per_client=1000,
                    batch_size=128, local_lr=0.05, seed=s)
        for name, kw in ((f'dp2_fedavg_eps{eps}_s{s}', dict(aggregator='fedavg')),
                         (f'dp2_aha_eps{eps}_s{s}',
                          dict(aggregator='adaptive', alpha=1.0,
                               beta=4.0, gamma=4.0))):
            try:
                run_one(OUT_DIR/f'{name}.json', **kw, **common)
                done+=1
            except Exception:
                fail+=1
                print(f'FAILED {name}'); traceback.print_exc()
    print(f'eps={eps} done ({done} ok, {fail} failed)', flush=True)
print(f'TOTAL {done} ok, {fail} failed')


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

CHANCE=1/9
print(f'chance = {CHANCE:.3f}; non-private AHA-Fed 0.702, FedAvg 0.679')
for eps in [1.0,10.0]:
    print(f'\n--- eps={eps}')
    n=0
    for key in ('accuracy','ece','auroc'):
        a=by_seed(f'runs/paper/real/dp2_aha_eps{eps}_s*.json',key)
        v=by_seed(f'runs/paper/real/dp2_fedavg_eps{eps}_s*.json',key)
        c=sorted(set(a)&set(v))
        if len(c)<2: continue
        n=len(c)
        x=[a[k] for k in c]; y=[v[k] for k in c]
        p=wilcoxon(x,y).pvalue
        better=sum(1 for i in range(len(c))
                   if (x[i]<y[i] if key=='ece' else x[i]>y[i]))
        print(f'  {key:9s} AHA {st.mean(x):.4f}  FedAvg {st.mean(y):.4f}  '
              f'p={p:.4f}  AHA better {better}/{len(c)}')
    if n:
        print(f'  (n={n}; min attainable two-sided p = {2/2**n:.4f})')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/dp2_results.zip runs/paper/real
!ls -lh /kaggle/working/dp2_results.zip
